# Proyecto 2 – Resultados
### Data Science

**Reto 18: Hackeando el cuerpo humano**. [HuBMAP - Hacking the Human Body](https://www.kaggle.com/competitions/hubmap-organ-segmentation)

Segmentación de unidades funcionales de tejido (FTU) en imágenes histológicas de cinco órganos: riñón, intestino grueso, pulmón, próstata y bazo.

Este notebook se trabaja en cadena, según `docs/division_avances_resultados.md`:

| Sección | Responsable |
| --- | --- |
| 1. Investigación y selección de modelos | Completada |
| 2. Pipeline de datos | Persona 1 |
| 3. Entrenamiento de los modelos | Persona 2 |
| 4. Evaluación, comparación y visualizaciones | Persona 3 |

El entrenamiento corre en Kaggle con GPU. Antes de pasar el turno, cada persona descarga el notebook ejecutado y lo sube al repositorio con su commit.

---
## Configuración inicial

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

# En Kaggle los datos están en /kaggle/input; en local, en data/
EN_KAGGLE = os.path.exists("/kaggle/input")
DATA_DIR = "/kaggle/input/hubmap-organ-segmentation/" if EN_KAGGLE else "data/"

---
## 1. Investigación y selección de modelos

Las referencias completas en formato APA están en `informe/referencias.bib`.

### 1.1 Revisión bibliográfica

**Contexto del problema**

| Referencia | Aporte al proyecto |
| --- | --- |
| Jain et al. (2023a), *Nature Communications* | Artículo oficial del reto. Describe el dataset (880 imágenes de HPA y HuBMAP, 351 de entrenamiento), la métrica (Dice promedio) y las soluciones ganadoras. |
| Jain et al. (2023b), *Communications Biology* | Competencia anterior de HuBMAP (solo glomérulos de riñón). Muestra que el modelo ganador se integró al portal de HuBMAP para segmentar a escala. |
| HuBMAP Consortium (2019), *Nature* | Objetivos del programa HuBMAP: construir un atlas de referencia del cuerpo humano a nivel celular. |
| de Bono et al. (2013), *Journal of Biomedical Semantics* | Definición de unidad funcional de tejido (FTU): bloque de células alrededor de un capilar, dentro de la distancia de difusión. |
| Bidanta et al. (2025), *Nature Communications* | Catálogo de 22 FTU en 10 órganos, con sus dimensiones y composición celular. |

**Aprendizaje profundo en imagen médica**

| Referencia | Aporte al proyecto |
| --- | --- |
| Litjens et al. (2017), *Medical Image Analysis* | Revisión de más de 300 trabajos. Las redes convolucionales dominan clasificación, detección y segmentación en imagen médica. |
| Siddique et al. (2021), *IEEE Access* | Revisión de U-Net y sus variantes (U-Net++, Attention U-Net, etc.) en distintas modalidades médicas. |
| Isensee et al. (2021), *Nature Methods* | nnU-Net: una U-Net bien configurada (preprocesamiento, entrenamiento, posprocesamiento) supera a arquitecturas más especializadas en 23 datasets. |

**Arquitecturas de segmentación**

| Referencia | Aporte al proyecto |
| --- | --- |
| Ronneberger et al. (2015), *MICCAI* | U-Net: encoder-decoder con conexiones de salto. Funciona con pocas imágenes gracias al data augmentation. |
| Zhou et al. (2020), *IEEE Transactions on Medical Imaging* | U-Net++: rediseña las conexiones de salto para combinar características de varias escalas. |
| Chen et al. (2018), *ECCV* | DeepLabV3+: convoluciones atrous y un decoder que mejora los bordes de los objetos. |
| Xie et al. (2021), *NeurIPS* | SegFormer: encoder transformer jerárquico con decoder MLP ligero, sin codificación posicional. |
| Xu et al. (2021), *ICCV* | CoaT: transformer con atención convolucional, usado como alternativa a SegFormer en el reto. |

**Preprocesamiento y función de pérdida**

| Referencia | Aporte al proyecto |
| --- | --- |
| Macenko et al. (2009), *IEEE ISBI* | Método clásico de normalización de tinción en histología. |
| Tellez et al. (2019), *Medical Image Analysis* | Compara augmentation de color y normalización de tinción con imágenes H&E de 4 órganos y 9 laboratorios. |
| Milletari et al. (2016), *3DV* | Propone la pérdida Dice para manejar el desbalance entre fondo y objeto. |

### 1.2 Hallazgos de las soluciones ganadoras del reto

Según Jain et al. (2023a):

- Los tres primeros lugares obtuvieron un Dice promedio de 0.835, 0.833 y 0.832 en el test privado.
- **Riñón** e **intestino grueso** fueron los órganos más fáciles; **pulmón** fue el más difícil (Dice ≈ 0.72 a 0.74), por la variación en la forma de los alvéolos.
- Las estrategias más efectivas fueron: ensambles con al menos un vision transformer, datos externos con pseudo-etiquetado, y data augmentation fuerte o normalización de tinción.
- **SegFormer** fue el mejor transformer. Como modelo individual (mit-b4) llegó a un Dice de 0.828, casi igual que el ensamble completo. **CoaT** rindió igual; **Swin** rindió mal.
- El tercer lugar adaptó el tamaño de píxel e hizo *histogram matching* entre imágenes de HPA y HuBMAP.

### 1.3 Selección de modelos


| Modelo | Tipo | Justificación |
| --- | --- | --- |
| U-Net con encoder ResNet34 | CNN (baseline) | Arquitectura de referencia en segmentación biomédica (Ronneberger et al., 2015; Siddique et al., 2021). nnU-Net muestra que sigue siendo muy competitiva (Isensee et al., 2021). |
| U-Net++ | CNN mejorada | Conexiones de salto multiescala, útiles porque el tamaño de las FTU varía mucho entre órganos (Zhou et al., 2020). |
| SegFormer | Transformer | Mejor modelo individual en el reto (Jain et al., 2023a; Xie et al., 2021). Permite comparar CNN contra transformer. |

Alternativas si algún modelo no funciona: DeepLabV3+ (Chen et al., 2018) o CoaT (Xu et al., 2021).

**Métricas de evaluación**

- **Dice** (métrica oficial del reto), promedio y por órgano.
- **IoU**, como métrica complementaria.
- Tiempo de entrenamiento y de inferencia.

**Función de pérdida sugerida:** Dice + BCE (Milletari et al., 2016).

**Preprocesamiento sugerido para la Persona 1:** normalización de tinción o augmentation de color (Macenko et al., 2009; Tellez et al., 2019) y ajuste del tamaño de píxel entre fuentes (Jain et al., 2023a).

---
## 2. Pipeline de datos

*Pendiente (Persona 1).*

### 2.1 Decodificación de máscaras RLE
### 2.2 Redimensionamiento y ajuste del tamaño de píxel
### 2.3 Normalización de tinción y data augmentation
### 2.4 División en entrenamiento, validación y prueba (estratificada por órgano)
### 2.5 Dataset y DataLoaders
### 2.6 Verificación visual del pipeline

---
## 3. Entrenamiento de los modelos

*Pendiente (Persona 2).*

### 3.1 Loop de entrenamiento común
### 3.2 U-Net (baseline)
### 3.3 U-Net++
### 3.4 SegFormer
### 3.5 Ajuste de hiperparámetros

---
## 4. Evaluación, comparación y visualizaciones

*Pendiente (Persona 3).*

### 4.1 Evaluación en el conjunto de prueba
### 4.2 Métricas por órgano
### 4.3 Visualizaciones
### 4.4 Discusión y selección del modelo